# NB_Purview_Copilot_BronzeToSilver
**Source:** OneLake **or** local CSV (notebook-only path below)  
**Targets (reads):** `okta_user_for_ai`, `hc_silver_historical`  
**Target:** `AI_Medallion.Silver.purview_copilot_usage`  
**Run after:** Okta + HC silver notebooks  

**Local CSV (recommended for large exports):** `PURVIEW_BRONZE_SOURCE = "local_csv"`.  
CSV header: `RecordId,CreationDate,RecordType,Operation,UserId,AuditData,AssociatedAdminUnits,AssociatedAdminUnitsNames`  
Loader uses **pandas → local parquet** (no Spark CSV, no `break_lineage_local` on 1M+ AuditData rows).  
Set **`PURVIEW_BRONZE_CREATION_MONTH`** in settings to filter **before** materialize (e.g. `"2026-09"`).

In [ ]:
import os
from pathlib import Path
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
os.environ.setdefault("MEDALLION_LOCAL_TABLE_CACHE", "1")
os.environ.setdefault("PYSPARK_DRIVER_MEMORY", "8g")
os.environ["MEDALLION_STAGING_SPARK_PARQUET"] = "0"

PURVIEW_BRONZE_SOURCE = "local_csv"
PURVIEW_BRONZE_CSV_DIR = Path.home() / "Downloads" / "purview_copilot_bronze"
PURVIEW_BRONZE_CSV_GLOB = "*.csv"
PURVIEW_BRONZE_CSV_CHUNK_ROWS = 50_000

# Filter CreationDate while loading CSV (strongly recommended for multi-million exports)
PURVIEW_BRONZE_CREATION_MONTH = "2026-09"
PURVIEW_BRONZE_FILENAMES = []


In [ ]:
%run ./Common/bootstrap


In [ ]:
# --- Purview local CSV bronze (notebook only; does not use Common/io break_lineage) ---
import json
import shutil

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

_PURVIEW_CSV_EXPORT_COLUMNS = [
    "RecordId",
    "CreationDate",
    "RecordType",
    "Operation",
    "UserId",
    "AuditData",
    "AssociatedAdminUnits",
    "AssociatedAdminUnitsNames",
]

_PURVIEW_BRONZE_COLUMNS = [
    "AuditData",
    "RecordId",
    "CreationDate",
    "FileName",
    "ExecutionDate",
    "UserId",
    "RecordType",
    "Operation",
    "AssociatedAdminUnits",
    "AssociatedAdminUnitsNames",
]


def _purview_bronze_parquet_dir(label: str = "purview_copilot_bronze") -> Path:
    return Path.home() / ".fabric" / "merge_staging" / "_break" / label


def _purview_merge_staging_dir(table_fqn: str) -> Path:
    return Path.home() / ".fabric" / "merge_staging" / table_fqn.replace(".", "__")


def _purview_normalize_csv_headers(df: pd.DataFrame) -> pd.DataFrame:
    rename = {}
    for col in df.columns:
        key = col.strip().lower().replace("_", "")
        mapping = {
            "recordid": "RecordId",
            "creationdate": "CreationDate",
            "recordtype": "RecordType",
            "operation": "Operation",
            "userid": "UserId",
            "auditdata": "AuditData",
            "associatedadminunits": "AssociatedAdminUnits",
            "associatedadminunitsnames": "AssociatedAdminUnitsNames",
        }
        if key in mapping and col != mapping[key]:
            rename[col] = mapping[key]
    if rename:
        df = df.rename(columns=rename)
    missing = [c for c in _PURVIEW_CSV_EXPORT_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"CSV missing columns {missing}; have {list(df.columns)}")
    return df


def _purview_execution_date(audit: str | None) -> str | None:
    if audit is None or (isinstance(audit, float) and pd.isna(audit)):
        return None
    try:
        return json.loads(str(audit)).get("CreationTime")
    except (json.JSONDecodeError, TypeError):
        return None


def _purview_parse_creation_ts(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series.astype(str).str.strip(), utc=True, errors="coerce")


def _purview_chunk_to_bronze(df: pd.DataFrame, source_file: str) -> pd.DataFrame:
    df = _purview_normalize_csv_headers(df)
    out = pd.DataFrame()
    out["AuditData"] = df["AuditData"].astype(str)
    out["RecordId"] = df["RecordId"].astype(str).str.strip()
    out["CreationDate"] = df["CreationDate"].astype(str).str.strip()
    out["FileName"] = source_file
    out["ExecutionDate"] = df["AuditData"].map(_purview_execution_date)
    out["UserId"] = df["UserId"].astype(str).str.strip()
    out["RecordType"] = df["RecordType"].astype(str)
    out["Operation"] = df["Operation"].astype(str)
    out["AssociatedAdminUnits"] = df["AssociatedAdminUnits"].astype(str)
    out["AssociatedAdminUnitsNames"] = df["AssociatedAdminUnitsNames"].astype(str)
    return out[_PURVIEW_BRONZE_COLUMNS]


def _purview_month_bounds(creation_month: str | None):
    if not creation_month:
        return None, None
    start = pd.Timestamp(f"{creation_month}-01", tz="UTC")
    if start.month == 12:
        end = pd.Timestamp(f"{start.year + 1}-01-01", tz="UTC")
    else:
        end = pd.Timestamp(f"{start.year}-{start.month + 1:02d}-01", tz="UTC")
    return start, end


def load_purview_bronze_from_local_csv(
    csv_dir: Path,
    *,
    glob_pattern: str = "*.csv",
    chunk_rows: int = 50_000,
    creation_month: str | None = None,
    filenames: list[str] | None = None,
    break_label: str = "purview_copilot_bronze",
):
    """Pandas chunked CSV → local parquet; Spark only reads file:// parquet at the end."""
    csv_dir = Path(csv_dir).expanduser().resolve()
    files = sorted(csv_dir.glob(glob_pattern))
    if filenames:
        allow = set(filenames)
        files = [p for p in files if p.name in allow]
    if not files:
        raise FileNotFoundError(f"No CSV files under {csv_dir} ({glob_pattern})")

    out_dir = _purview_bronze_parquet_dir(break_label)
    shutil.rmtree(out_dir, ignore_errors=True)
    out_dir.mkdir(parents=True, exist_ok=True)

    month_start, month_end = _purview_month_bounds(creation_month)
    if month_start is not None:
        print(f"purview CSV load: CreationDate in {creation_month}", flush=True)

    print(f"purview CSV load: {len(files)} file(s) → {out_dir}", flush=True)
    for p in files:
        print(f"  - {p.name}", flush=True)

    part_idx = 0
    total_rows = 0
    for csv_path in files:
        reader = pd.read_csv(
            csv_path,
            chunksize=chunk_rows,
            dtype=str,
            keep_default_na=False,
            encoding="utf-8-sig",
        )
        for chunk in reader:
            bronze_chunk = _purview_chunk_to_bronze(chunk, csv_path.name)
            if month_start is not None:
                ts = _purview_parse_creation_ts(bronze_chunk["CreationDate"])
                mask = (ts >= month_start) & (ts < month_end)
                bronze_chunk = bronze_chunk.loc[mask]
                if bronze_chunk.empty:
                    continue
            total_rows += len(bronze_chunk)
            pq.write_table(
                pa.Table.from_pandas(bronze_chunk, preserve_index=False),
                out_dir / f"part-{part_idx:05d}.parquet",
            )
            part_idx += 1
            if part_idx % 10 == 0:
                print(f"purview CSV load: {total_rows} rows written…", flush=True)

    if part_idx == 0:
        raise RuntimeError(
            "No rows written after CSV load (check PURVIEW_BRONZE_CREATION_MONTH / filenames)."
        )

    print(f"purview CSV load: done — {total_rows} rows, {part_idx} parquet part(s)", flush=True)
    return spark.read.parquet(str(out_dir))


def purview_publish_merge_staging_notebook(df, table_fqn: str) -> Path:
    """Write silver staging parquet for merge_incremental (notebook-only; small post-join data)."""
    apply_local_merge_spark_conf(spark)
    path = _purview_merge_staging_dir(table_fqn)
    shutil.rmtree(path, ignore_errors=True)
    print(f"purview merge staging: {table_fqn} -> {path}", flush=True)
    df.coalesce(1).write.mode("overwrite").parquet(str(path))
    return path


In [ ]:
if PURVIEW_BRONZE_SOURCE == "local_csv":
    bronze = load_purview_bronze_from_local_csv(
        PURVIEW_BRONZE_CSV_DIR,
        glob_pattern=PURVIEW_BRONZE_CSV_GLOB,
        chunk_rows=PURVIEW_BRONZE_CSV_CHUNK_ROWS,
        creation_month=PURVIEW_BRONZE_CREATION_MONTH,
        filenames=PURVIEW_BRONZE_FILENAMES or None,
    )
elif PURVIEW_BRONZE_SOURCE == "onelake":
    bronze = read_table_broken_lineage(
        TABLE_PURVIEW_COPILOT_BRONZE,
        "purview_copilot_bronze",
    )
else:
    raise ValueError(f"Unknown PURVIEW_BRONZE_SOURCE: {PURVIEW_BRONZE_SOURCE!r}")


In [ ]:
okta = read_table_broken_lineage(TABLE_OKTA_SILVER, "okta_silver")


In [ ]:
hc = read_table_broken_lineage(TABLE_HC_SILVER, "hc_silver")


In [ ]:
apply_local_merge_spark_conf(spark)

if PURVIEW_BRONZE_SOURCE == "local_csv" and PURVIEW_BRONZE_CREATION_MONTH:
    min_date = f"{PURVIEW_BRONZE_CREATION_MONTH}-01"
    print(f"purview bronze: CreationDate month already applied at CSV load ({PURVIEW_BRONZE_CREATION_MONTH})", flush=True)
elif PURVIEW_BRONZE_CREATION_MONTH:
    _creation_col = (
        PURVIEW_BRONZE_FILTER_DATE_COLUMN
        if PURVIEW_BRONZE_FILTER_DATE_COLUMN in bronze.columns
        else "creation_date"
    )
    _month_start = F.to_date(F.lit(f"{PURVIEW_BRONZE_CREATION_MONTH}-01"))
    _month_end = F.add_months(_month_start, 1)
    _creation_date = F.to_timestamp(F.trim(F.col(_creation_col))).cast("date")
    bronze = bronze.filter((_creation_date >= _month_start) & (_creation_date < _month_end))
    min_date = f"{PURVIEW_BRONZE_CREATION_MONTH}-01"
else:
    min_date = os.environ.get(
        "MEDALLION_PURVIEW_MIN_CREATION_DATE", PURVIEW_COPILOT_MIN_CREATION_DATE
    )

if PURVIEW_BRONZE_FILENAMES and PURVIEW_BRONZE_SOURCE != "local_csv":
    _file_col = "FileName" if "FileName" in bronze.columns else "file_name"
    bronze = bronze.filter(F.col(_file_col).isin(PURVIEW_BRONZE_FILENAMES))

print(f"purview_copilot_usage: creation_date >= {min_date}")

write_df = build_purview_copilot_usage_silver(
    bronze,
    okta,
    hc,
    min_creation_date=min_date,
)

if PURVIEW_BRONZE_SOURCE == "local_csv":
    purview_publish_merge_staging_notebook(write_df, TABLE_PURVIEW_COPILOT_SILVER)
else:
    write_df = break_lineage_local(write_df, "purview_copilot_silver")
    publish_merge_staging(write_df, TABLE_PURVIEW_COPILOT_SILVER)


In [ ]:
merge_incremental(TABLE_PURVIEW_COPILOT_SILVER, PURVIEW_COPILOT_MERGE_KEY)
